# Deeptech M&A Momentum: Optimal Lag Determination
 
## Phase 3, Step 3.3: Optimal Lag Determination (Signal Testing)

This notebook performs the crucial statistical analysis to find the predictive relationship between the transformed M&A Volume features (from Step 3.2) and future sector returns. We use the **Granger Causality Test** to determine the optimal M&A frequency ('1mo', '3mo', or '6mo') and the best lead-time (lag) that provides a statistically significant signal for predicting sector rotation.
 
**Test Matrix:** For each sector, we test 18 predictive combinations (3 Frequencies x 6 Lags).

---

In [1]:
# --- 1. Standard Library Imports ---
from pathlib import Path
import sys
from typing import Dict, List, Tuple

# --- 2. Third-Party Library Imports ---
import polars as pl
import pandas as pd # Required for statsmodels compatibility
from statsmodels.tsa.stattools import grangercausalitytests

In [2]:
# --- Configuration ---
MNA_INPUT_DIR = Path("../../data/processed")
RESULTS_OUTPUT_DIR = Path("../../data/outputs")
RESULTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True) # Ensure outputs directory exists
RESULTS_OUTPUT_PATH = RESULTS_OUTPUT_DIR / "3.3_granger_causality_results.csv"

MAX_GRANGER_LAG = 6 # Max lag to test (1 to 6 months)
CRITICAL_P_VALUE = 0.05

# Predictor columns from the MASTER file generated in 3.2
SCALED_VOLUME_COLS = ["Volume_MNA_Scaled_1mo", "Volume_MNA_Scaled_3mo", "Volume_MNA_Scaled_6mo"]

# List of Tickers to analyze (excluding the benchmark ^GSPC)
SECTOR_TICKERS = list({
    "HYDR", "IYZ", "TMET", "BOTZ", "ITA", "SNSR", "AIQ", "SOXX", 
    "LIT", "PRNT", "GRID", "ARKG", "KRBN", "XLB", "IYJ", "FAN"
})


### Step 1: Granger Causality Function Definition

We define the function that handles the creation of lagged features and execution of the statistical test.

In [3]:
def run_granger_tests(df_data: pl.DataFrame) -> List[Dict]:
    """
    Performs Granger Causality tests for all scaled volume features (1m, 3m, 6m) 
    against Returns_Target across 6 lags for all deeptech sectors.
    """
    all_granger_results = []
    
    # Filter out the benchmark and sort
    df_data = df_data.filter(pl.col("Ticker").is_in(SECTOR_TICKERS)).sort("Ticker", "Date")
    
    for ticker in SECTOR_TICKERS:
        df_sector = df_data.filter(pl.col("Ticker") == ticker)
        
        # Determine if there is sufficient data for the largest lag test
        if len(df_sector) < 2 * MAX_GRANGER_LAG + 10:
            print(f"Skipping {ticker}: Not enough data points ({len(df_sector)}).")
            continue
            
        print(f"\n--- Testing Ticker: {ticker} ---")
        
        # Iterate over all three scaled volume features
        for volume_col in SCALED_VOLUME_COLS:
            
            # Start with a clean selection of the Target and the current Volume feature
            df_test = df_sector.select(["Date", "Returns_Target", volume_col]).sort("Date")
            
            # 1. Create lagged features (Shift the volume column)
            for lag in range(1, MAX_GRANGER_LAG + 1):
                lagged_col = f"{volume_col}_Lag{lag}"
                
                # Apply the lag (shift the predictor column backwards in time)
                df_test = df_test.with_columns(
                    pl.col(volume_col).shift(lag).alias(lagged_col)
                )

            # Convert to Pandas for statsmodels
            # We select the target and the current volume feature up to the max lag (inclusive)
            cols_to_select = ["Returns_Target"] + [f"{volume_col}_Lag{lag}" for lag in range(1, MAX_GRANGER_LAG + 1)]
            df_test_pd = df_test.select(cols_to_select).to_pandas().dropna()
            
            # 2. Run Granger Causality Test
            try:
                # GC tests if all lags up to MAX_GRANGER_LAG jointly cause the returns.
                # The function internally tests all lags up to maxlag.
                gc_test = grangercausalitytests(
                    df_test_pd, 
                    maxlag=MAX_GRANGER_LAG, 
                    verbose=False
                )
            except Exception as e:
                print(f"Error running GC for {ticker} ({volume_col}): {e}")
                continue

            # 3. Extract results for each individual lag
            for lag in range(1, MAX_GRANGER_LAG + 1):
                # Use the F-test p-value (index [1] of the ssr_ftest tuple)
                p_value = gc_test[lag][0]['ssr_ftest'][1]
                is_causal = p_value <= CRITICAL_P_VALUE
                
                all_granger_results.append({
                    'Ticker': ticker,
                    'Predictor_Freq': volume_col.split('_')[-1],
                    'Lag_Months': lag,
                    'F_Test_PValue': p_value,
                    'Causal_Relation': is_causal
                })
                # Log strong signals
                if is_causal:
                     print(f"  -> {volume_col} | Lag {lag}: {p_value:.4f} (CAUSAL)")

    return all_granger_results

### Step 2: Execution and Saving Results

We load the single master feature file and execute the tests, saving the results to `data/outputs/`.

In [4]:
# --- Execution ---

INPUT_PATH = MNA_INPUT_DIR / "3.2_aligned_features_MASTER.csv"
    
if not INPUT_PATH.exists():
    print(f"ERROR: Master Feature CSV not found at {INPUT_PATH}. Please run 3.2 first.")
else:
    # Load the master file
    df_master = pl.read_csv(INPUT_PATH)
    
    # Run the tests
    results = run_granger_tests(df_master)
    
    # Save the results
    df_granger_results = pd.DataFrame(results)
    df_granger_results.to_csv(RESULTS_OUTPUT_PATH, index=False)
    
    print("\n" + "=" * 80)
    print(f"✓ All Granger Causality tests complete.")
    print(f"Results saved to {RESULTS_OUTPUT_PATH}")
    print("=" * 80)


--- Testing Ticker: PRNT ---
Error running GC for PRNT (Volume_MNA_Scaled_1mo): wrong shape for coefs
Error running GC for PRNT (Volume_MNA_Scaled_3mo): wrong shape for coefs
Error running GC for PRNT (Volume_MNA_Scaled_6mo): wrong shape for coefs

--- Testing Ticker: LIT ---
Error running GC for LIT (Volume_MNA_Scaled_1mo): wrong shape for coefs
Error running GC for LIT (Volume_MNA_Scaled_3mo): wrong shape for coefs
Error running GC for LIT (Volume_MNA_Scaled_6mo): wrong shape for coefs

--- Testing Ticker: IYJ ---
Error running GC for IYJ (Volume_MNA_Scaled_1mo): wrong shape for coefs
Error running GC for IYJ (Volume_MNA_Scaled_3mo): wrong shape for coefs
Error running GC for IYJ (Volume_MNA_Scaled_6mo): wrong shape for coefs

--- Testing Ticker: BOTZ ---
Error running GC for BOTZ (Volume_MNA_Scaled_1mo): wrong shape for coefs
Error running GC for BOTZ (Volume_MNA_Scaled_3mo): wrong shape for coefs
Error running GC for BOTZ (Volume_MNA_Scaled_6mo): wrong shape for coefs
Skipping TME

c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print 